# Colab Pro: fine-tune V-JEPA encoder rồi train online AC predictor

Notebook này dành cho experiment lớn trên Colab Pro/RTX Pro 6000 96GB VRAM.

Pipeline:

1. Mount Google Drive và mở repo `NN-JEPA`.
2. Cài package từ repo hiện tại.
3. Phase 1: domain-adapt/fine-tune encoder bằng JEPA-style EMA teacher trên frame RC.
4. Phase 2: train online AC predictor bằng encoder đã fine-tune.

Lưu ý quan trọng:

- Đây là experiment riêng, không thay thế baseline feature-cache ổn định hiện tại.
- Phase 1 không phải full V-JEPA pretraining của Meta. Nó là domain adaptation nhẹ: student encoder học dự đoán latent frame kế tiếp của EMA teacher, có anchor loss để hạn chế drift.
- Phase 2 mặc định freeze encoder đã fine-tune rồi train predictor online. Nếu muốn end-to-end tiếp, bật `--phase2-train-encoder`, nhưng rủi ro overfit/drift cao hơn.
- Notebook đọc `source_frame_path` raw frame, resize trực tiếp về 384px, không dùng ảnh processed 224px.


## 1. Mount Drive và cấu hình đường dẫn

Sửa các biến dưới đây cho đúng vị trí repo/data/checkpoint trên Google Drive của bạn.

Khuyến nghị cho RTX Pro 6000 96GB:

- Bắt đầu với `vit_large_384` nếu đã có checkpoint ViT-L.
- Nếu OOM hoặc quá chậm, đổi về `vit_base_384`.
- `vit_giant_384`/`vit_gigantic_384` cần xformers và rất nặng, chỉ thử sau khi ViT-L chạy ổn.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Không chạy trong Colab hoặc Drive đã mount:', exc)

# Sửa đường dẫn này nếu repo nằm chỗ khác trên Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/NN-JEPA')
os.chdir(PROJECT_DIR)
print('cwd =', Path.cwd())

# Data mix hiện tại.
MANIFEST_DIR = PROJECT_DIR / 'data/experiments/servo_old_mix_v1/processed/manifests'

# Chọn encoder/checkpoint.
ENCODER = 'vit_large_384'  # đổi thành 'vit_base_384' nếu muốn an toàn hơn
VJEPA_CHECKPOINT = PROJECT_DIR / 'checkpoints/vjepa2_1/vjepa2_1_vitl_dist_vitG_384.pt'
CHECKPOINT_KEY = 'ema_encoder'

# Nếu dùng ViT-B thì dùng 3 dòng này thay cho cấu hình ViT-L:
# ENCODER = 'vit_base_384'
# VJEPA_CHECKPOINT = PROJECT_DIR / 'checkpoints/vjepa2_1/vjepa2_1_vitb_dist_vitG_384.pt'
# CHECKPOINT_KEY = 'ema_encoder'

OUTPUT_DIR = PROJECT_DIR / 'checkpoints/colab_vitl_encoder_online_ac_mix_v1'
VJEPA_ROOT = PROJECT_DIR / 'vjepa2'

# Nếu manifest được tạo trên PC Ubuntu, path có thể bắt đầu bằng prefix này.
# Script sẽ rewrite sang PROJECT_DIR để đọc được trên Colab/Drive.
PATH_REWRITE_FROM = '/home/heheboiz/data/NN-JEPA'
PATH_REWRITE_TO = str(PROJECT_DIR)

print('manifest exists:', MANIFEST_DIR.exists(), MANIFEST_DIR)
print('checkpoint exists:', VJEPA_CHECKPOINT.exists(), VJEPA_CHECKPOINT)
print('vjepa root exists:', VJEPA_ROOT.exists(), VJEPA_ROOT)


## 2. Cài dependency và kiểm tra GPU

Nếu Colab restart runtime, chạy lại từ cell này trở xuống.


In [ ]:
!pip install -q -e . hydra-core timm einops tqdm wandb

import torch, subprocess, sys
print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

!PYTHONPATH=src python3 -m tools.train_colab_encoder_online_ac --help | head -80


## 3. Hàm chạy command

Tất cả tham số được để thành list để dễ sửa. Không cần nhớ lệnh dài.


In [ ]:
import subprocess, os

def run_nn_jepa(args):
    env = os.environ.copy()
    env['PYTHONPATH'] = 'src'
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print(' '.join(args))
    return subprocess.run(args, check=True, env=env)

COMMON_ARGS = [
    'python3', '-m', 'tools.train_colab_encoder_online_ac',
    '--manifest-dir', str(MANIFEST_DIR),
    '--path-rewrite-from', PATH_REWRITE_FROM,
    '--path-rewrite-to', PATH_REWRITE_TO,
    '--vjepa-root', str(VJEPA_ROOT),
    '--vjepa-checkpoint', str(VJEPA_CHECKPOINT),
    '--checkpoint-key', CHECKPOINT_KEY,
    '--encoder', ENCODER,
    '--image-size', '384',
    '--raw-frames-per-sample', '8',
    '--frame-stride', '2',
    '--sequence-stride', '1',
    '--auto-steps', '2',
    '--amp-dtype', 'bf16',
    '--num-workers', '4',
    '--prefetch-factor', '2',
    '--output-dir', str(OUTPUT_DIR),
]

print('COMMON_ARGS ready')


## 4. Smoke test nhỏ trước khi train thật

Cell này chỉ chạy 1 epoch phase 1 trên ít window để kiểm tra path/checkpoint/GPU. Nên chạy trước khi train full.


In [ ]:
SMOKE_OUTPUT_DIR = OUTPUT_DIR.parent / (OUTPUT_DIR.name + '_smoke')
smoke_args = COMMON_ARGS + [
    '--phase', 'finetune_encoder',
    '--output-dir', str(SMOKE_OUTPUT_DIR),
    '--batch-size', '1',
    '--eval-batch-size', '1',
    '--max-train-windows', '32',
    '--max-val-windows', '8',
    '--phase1-epochs', '1',
    '--phase1-train-last-n-blocks', '2',
]
run_nn_jepa(smoke_args)


## 5. Train full: Phase 1 fine-tune encoder + Phase 2 online predictor

Default dưới đây khá bảo thủ cho ViT-L 384. Với 96GB VRAM, nếu GPU còn rảnh có thể tăng `--batch-size` từ `4` lên `6` hoặc `8`.

Phase 1:

- Train last 4 block của encoder.
- LR encoder rất nhỏ `1e-6`.
- Head LR `1e-4`.
- EMA teacher decay `0.996`.

Phase 2:

- Dùng checkpoint `encoder_finetune_best.pt`.
- Mặc định freeze encoder, train online predictor `official_lite base`.
- Nếu muốn train tiếp encoder trong phase 2, thêm flag `--phase2-train-encoder`, nhưng không khuyến nghị ở lần chạy đầu.


In [ ]:
full_args = COMMON_ARGS + [
    '--phase', 'all',
    '--batch-size', '4',
    '--eval-batch-size', '2',
    '--phase1-epochs', '5',
    '--phase1-encoder-lr', '1e-6',
    '--phase1-head-lr', '1e-4',
    '--phase1-train-last-n-blocks', '4',
    '--phase1-anchor-weight', '0.25',
    '--phase1-ema-decay', '0.996',
    '--phase2-epochs', '100',
    '--phase2-predictor-lr', '5e-5',
    '--predictor-type', 'official_lite',
    '--model-size', 'base',
    '--warmup-epochs', '4',
    '--early-stopping-patience', '15',
]
run_nn_jepa(full_args)


## 6. Nếu Colab bị ngắt: resume phase 2

Nếu đã fine-tune encoder xong và phase 2 bị ngắt, chạy cell này để resume từ `online_ac_last.pt`.

Nếu phase 1 bị ngắt giữa chừng, hiện script chưa resume phase 1; chạy lại phase 1 từ đầu hoặc giảm `phase1_epochs`.


In [ ]:
resume_args = COMMON_ARGS + [
    '--phase', 'train_predictor',
    '--phase2-encoder-checkpoint', str(OUTPUT_DIR / 'encoder_finetune_best.pt'),
    '--phase2-resume-from', str(OUTPUT_DIR / 'online_ac_last.pt'),
    '--batch-size', '4',
    '--eval-batch-size', '2',
    '--phase2-epochs', '100',
    '--phase2-predictor-lr', '5e-5',
    '--predictor-type', 'official_lite',
    '--model-size', 'base',
]
# Bỏ comment dòng dưới nếu cần resume.
# run_nn_jepa(resume_args)


## 7. Output quan trọng

Sau khi chạy, các file quan trọng nằm trong `OUTPUT_DIR`:

- `encoder_finetune_best.pt`: encoder fine-tuned tốt nhất theo val phase 1.
- `encoder_finetune_last.pt`: encoder phase 1 epoch cuối.
- `online_ac_best.pt`: predictor online tốt nhất theo val phase 2.
- `online_ac_last.pt`: checkpoint để resume phase 2.

Khi so với baseline feature-cache hiện tại, phải so cùng val split và xem tối thiểu:

- `val/loss` hoặc loss JSON in ra mỗi epoch.
- rollout@1 ratio, rollout@3 ratio nếu sau này thêm eval script cho checkpoint online này.
- Quan trọng nhất: offline planning/dry-run trước khi đưa lên xe thật.
